# Source-supported posterior: finite witnesses

See [theory_main.md](theory_main.md). This notebook verifies algebra and the common-view identification counterexample. It does not train an HSE/LLapDiff model or evaluate industrial diagnosis.

In [ ]:
import csv, os
from pathlib import Path
import numpy as np
rows=[]
def record(name,value):
    rows.append(dict(witness=name,value=float(value),scope='finite_theory_only'))
    print(name, float(value))
I=np.eye(4)
Oa=np.diag([1.,1.,0.,0.]); Ob=np.diag([1.,0.,1.,0.])
Us=np.diag([1.,1.,1.,0.]); Cs=np.diag([1.,0.,0.,0.])
blocks=[Cs,Oa-Cs,Us-Oa,I-Us]
np.testing.assert_allclose(sum(blocks),I)
for j,Pj in enumerate(blocks):
    np.testing.assert_allclose(Pj@Pj,Pj)
    for k,Q in enumerate(blocks):
        if j!=k: np.testing.assert_allclose(Pj@Q,0.)
record('source_partition_sum_residual',np.linalg.norm(sum(blocks)-I))
record('source_role_rank_each',min(np.trace(Pj) for Pj in blocks))
Ot=np.diag([0.,1.,1.,0.]); Ct=np.zeros((4,4))
np.testing.assert_allclose(sum([Ct,Ot-Ct,Us-Ot,I-Us]),I)
record('target_common_rank',np.trace(Ct))
record('target_missing_rank',np.trace(Us-Ot))

## Sensitivity and overlapping source views
A positive coordinate sensitivity is not individual recoverability. The second example uses two actual source pair laws, (C,P) and (C,M), not merely separate one-dimensional marginals.

In [ ]:
A=np.array([[1.,1.]])
a=np.array([2.,3.]); b=a+np.array([1.,-1.])
assert np.all(np.diag(A.T@A)>0)
np.testing.assert_allclose(A@a,A@b)
record('mixed_operator_ambiguous_state_distance',np.linalg.norm(a-b))
record('mixed_operator_observation_difference',np.linalg.norm(A@a-A@b))
# Axis order is (C,P,M). C is independent of the fair pair's generating bit.
pm_plus=np.eye(2)/2; pm_minus=np.fliplr(np.eye(2))/2
joint_plus=np.stack([pm_plus/2,pm_plus/2])
joint_minus=np.stack([pm_minus/2,pm_minus/2])
for joint in [joint_plus,joint_minus]:
    np.testing.assert_allclose(joint.sum(),1.)
# Source A sees (C,P), source B sees (C,M): both observed joint laws coincide.
np.testing.assert_allclose(joint_plus.sum(axis=2),joint_minus.sum(axis=2))
np.testing.assert_allclose(joint_plus.sum(axis=1),joint_minus.sum(axis=1))
# Their common-view conditional p(M|C) agrees, but the full-input conditional does not.
np.testing.assert_allclose(joint_plus.sum(axis=1)/.5,joint_minus.sum(axis=1)/.5)
conditional_plus=joint_plus[1,1,1]/joint_plus[1,1,:].sum()
conditional_minus=joint_minus[1,1,1]/joint_minus[1,1,:].sum()
conditional_gap=abs(conditional_plus-conditional_minus)
assert conditional_gap==1
record('unpaired_same_marginal_conditional_gap',conditional_gap)

## Source-null likelihood does not imply prior independence

In [ ]:
rho=.8; prior=np.array([[1.,rho],[rho,1.]])
operator=np.array([[1.,0.]])
gain=prior@operator.T/(float((operator@prior@operator.T).item())+1)
mean=gain[:,0]
post=prior-gain@operator@prior
np.testing.assert_allclose(mean,[.5,.4])
np.testing.assert_allclose(post[1,1],.68)
record('correlated_prior_null_posterior_mean',mean[1])
record('correlated_prior_null_posterior_variance',post[1,1])
independent=np.eye(2); gain_i=independent@operator.T/2
post_i=independent-gain_i@operator@independent
record('independent_prior_null_posterior_variance',post_i[1,1])
assert post_i[1,1]==1

## Projection preserves scope, not posterior correctness

In [ ]:
G=np.diag([0.,0.,1.,0.]); observed=np.array([.4,-.3,0.,0.])
rng=np.random.default_rng(18); v=G@rng.normal(size=4)
leak=[]; drift=[]
for k in range(100):
    v=G@(.9*v+rng.normal(size=4))
    state=observed+v
    leak.append(np.linalg.norm((I-G)@v))
    drift.append(np.linalg.norm(Oa@(state-observed)))
record('projected_update_max_forbidden_energy',max(leak))
record('projected_update_max_observed_drift',max(drift))
assert max(leak)==0 and max(drift)==0
G0=np.zeros((4,4))
recovered=None if np.linalg.matrix_rank(G0)==0 else G0@rng.normal(size=4)
assert recovered is None
record('empty_eligibility_generated_coordinates',0)
# A target can reverse the coupling even after the source joint has been identified.
record('target_conditional_reversal_gap',abs(conditional_plus-conditional_minus))

## Finite outputs
The values remain comparable to the retained 14-row witness. The stronger overlapping-source example has the same conditional gap, not a new empirical performance score.

In [ ]:
out=Path(os.environ.get('TII_BUILD_OUTPUT','outputs/tii_support'))
out.mkdir(parents=True,exist_ok=True)
with (out/'support_witness.csv').open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=['witness','value','scope'])
    writer.writeheader(); writer.writerows(rows)
print('SUPPORT_POSTERIOR_FINITE_WITNESS_PASS',len(rows))